# Reto 2 — Modelado de Datos en MongoDB

**Máster NTIC — Bases de Datos NoSQL**

---

## Objetivos

1. **Diseñar e implementar** un modelo de datos embebido en MongoDB para locales, actividades, licencias y terrazas de Madrid.
2. **Ejecutar consultas** de agregación para validar el modelo.
3. **Crear índices** (simple, compuesto y de array) y analizar su impacto.
4. **Extender el modelo (v2)** incorporando los alojamientos de Airbnb.
5. **Validar el modelo extendido** con consultas cruzadas.

> **Pre-requisito:** Ejecutar el notebook `01_data_preparation.ipynb` para generar los ficheros `data/processed/locales_mongodb.json` y `data/processed/listings_mongodb.json`.
>
> **Conexión MongoDB:** Se asume MongoDB corriendo en `localhost:27017` (instalación local o Docker). Para Docker: `docker run -d -p 27017:27017 --name mongo mongo:7`

## 1. Instalación e importación de librerías

In [2]:
import json
import time
from pathlib import Path

import pymongo
from pymongo import MongoClient

print(f"PyMongo version: {pymongo.__version__}")

PyMongo version: 4.16.0


## 2. Configuración y conexión a MongoDB

In [3]:
MONGO_URI        = "mongodb://localhost:27017"
DB_NAME          = "ucm_locales_alojamientos"
COL_LOCALES      = "locales"
COL_LISTINGS     = "listings"

client = MongoClient(MONGO_URI)
db     = client[DB_NAME]

print(f"Conectado a MongoDB: {MONGO_URI}")
print(f"Base de datos: {DB_NAME}")

Conectado a MongoDB: mongodb://localhost:27017
Base de datos: ucm_locales_alojamientos


## 3. Diseño del modelo de datos — Versión 1

### 3.1 Propuesta de modelo (patrón Embebido)

Se adopta el patrón **Documento Embebido** (Embedded Document Pattern):  
cada documento `local` contiene embebidos los subdocumentos de sus actividades económicas, licencias y terraza.

**Justificación:**
- Las actividades, licencias y terrazas no tienen sentido fuera del contexto de un local.
- La cardinalidad es manejable (pocos registros por local: tipicamente 1-3 actividades, 1-2 licencias, 0-1 terrazas).
- Permite recuperar toda la información de un establecimiento en una sola lectura, sin joins.

**Estructura del documento:**
```json
{
  "id_local": 12345,
  "rotulo": "Cafetería Ejemplo",
  "desc_distrito_local": "CENTRO",
  "desc_barrio_local": "SOL",
  "location": { "type": "Point", "coordinates": [-3.703, 40.416] },
  "hora_apertura1": "08:00",
  "hora_cierre2": "22:00",
  "actividades": [
    { "id_seccion": "I", "desc_seccion": "HOSTELERIA",
      "id_division": "55", "desc_division": "RESTAURANTES",
      "id_epigrafe": "672", "desc_epigrafe": "CAFES Y BARES" }
  ],
  "licencias": [
    { "ref_licencia": "2023/12345",
      "desc_tipo_licencia": "LICENCIA DE APERTURA",
      "desc_tipo_situacion_licencia": "Concedida",
      "fecha_dec_lic": "2023-01-15" }
  ],
  "terraza": {
    "id_terraza": 9876,
    "mesas_es": 6,
    "sillas_es": 24,
    "desc_tipo_situacion_terraza": "Autorizada"
  }
}
```

> El diseño detallado, el diagrama y la justificación completa están en el **README.md** del repositorio.

### 3.2 Implementación del modelo — Carga de datos en MongoDB

In [4]:
processed_dir = Path("../data/processed")

locales_file  = processed_dir / "locales_mongodb.json"
listings_file = processed_dir / "listings_mongodb.json"

assert locales_file.exists(),  f"Fichero no encontrado: {locales_file}"
assert listings_file.exists(), f"Fichero no encontrado: {listings_file}"

print(f"locales_mongodb.json  : {locales_file.stat().st_size / 1024**2:.1f} MB")
print(f"listings_mongodb.json : {listings_file.stat().st_size / 1024**2:.1f} MB")

locales_mongodb.json  : 200.9 MB
listings_mongodb.json : 9.6 MB


In [5]:
def cargar_coleccion(db, nombre_col, fichero_json, drop_first=True):
    """Carga un fichero JSON (array) en una colección MongoDB."""
    col = db[nombre_col]
    if drop_first:
        col.drop()
        print(f"Colección '{nombre_col}' eliminada.")

    with open(fichero_json, encoding="utf-8") as f:
        datos = json.load(f)

    if isinstance(datos, list) and datos:
        resultado = col.insert_many(datos)
        print(f"Insertados {len(resultado.inserted_ids):,} documentos en '{nombre_col}'.")
    else:
        print(f"Fichero vacío o formato incorrecto: {fichero_json}")

    return col


col_locales  = cargar_coleccion(db, COL_LOCALES,  locales_file)
col_listings = cargar_coleccion(db, COL_LISTINGS, listings_file)

Colección 'locales' eliminada.
Insertados 27,402 documentos en 'locales'.
Colección 'listings' eliminada.
Insertados 16,313 documentos en 'listings'.


In [6]:
print(f"Total locales   : {col_locales.count_documents({}):,}")
print(f"Total listings  : {col_listings.count_documents({}):,}")

Total locales   : 27,402
Total listings  : 16,313


## 4. Consultas sobre el modelo (v1)

### 4a. Total de locales y terrazas por distrito y barrio

Se añade top 10, para tener controlado el output de la query, y evitar que la consulta devuelva cientos de registros.

In [32]:
pipeline_4a = [
    {
        "$group": {
            "_id": {
                "distrito": "$desc_distrito_local",
                "barrio":   "$desc_barrio_local"
            },
            "total_locales":  {"$sum": 1},
            "total_terrazas": {
                "$sum": {"$cond": [{"$ne": ["$terraza.id_terraza", None]}, 1, 0]}
            }
        }
    },
    {"$sort": {"total_locales": -1}},
    {"$limit": 10},
    {
        "$project": {
            "_id":            0,
            "distrito":       "$_id.distrito",
            "barrio":         "$_id.barrio",
            "total_locales":  1,
            "total_terrazas": 1
        }
    }
]

resultados_4a = list(col_locales.aggregate(pipeline_4a))
print("Total de locales y terrazas por distrito y barrio (top 15):")
for r in resultados_4a:
    print(f"  {r['distrito']:<25} | {r['barrio']:<30} | locales: {r['total_locales']:>4} | terrazas: {r['total_terrazas']:>3}")

Total de locales y terrazas por distrito y barrio (top 15):
  VILLAVERDE                | SAN ANDRES                     | locales:  831 | terrazas:  11
  VILLA DE VALLECAS         | CASCO H.VALLECAS               | locales:  702 | terrazas:  13
  CENTRO                    | EMBAJADORES                    | locales:  638 | terrazas:  28
  CIUDAD LINEAL             | PUEBLO NUEVO                   | locales:  627 | terrazas:  15
  CARABANCHEL               | VISTA ALEGRE                   | locales:  568 | terrazas:  17
  LATINA                    | PUERTA DEL ANGEL               | locales:  537 | terrazas:   9
  PUENTE DE VALLECAS        | SAN DIEGO                      | locales:  536 | terrazas:   8
  CENTRO                    | UNIVERSIDAD                    | locales:  499 | terrazas:  22
  LATINA                    | ALUCHE                         | locales:  492 | terrazas:  28
  FUENCARRAL-EL PARDO       | VALVERDE                       | locales:  488 | terrazas:  30


### 4b. Tipos de licencia y cantidad por tipo

In [28]:
pipeline_4b = [
    {"$unwind": {"path": "$licencias", "preserveNullAndEmptyArrays": False}},
    {
        "$group": {
            "_id":    "$licencias.desc_tipo_licencia",
            "cantidad": {"$sum": 1}
        }
    },
    {"$match": {"_id": {"$ne": None}}},
    {"$sort": {"cantidad": -1}},
    {"$project": {"_id": 0, "tipo_licencia": "$_id", "cantidad": 1}}
]

resultados_4b = list(col_locales.aggregate(pipeline_4b))
print("Tipos de licencia y cantidad:")
for r in resultados_4b:
    print(f"  {r.get('tipo_licencia','(sin tipo)'):<50} → {r['cantidad']:>6}")

Tipos de licencia y cantidad:
  Transmisión de licencia Urbanística                →  13156
  Declaración Responsable                            →  12412
  Licencia Urbanística                               →   7135
  Licencia de Funcionamiento                         →   1491
  Licencia recogida en el trabajo de campo           →   1285


### 4c. Locales y terrazas con licencia "En trámite"

In [31]:
import re

pipeline_4c = [
    # Filtrar documentos que tengan al menos una licencia que empiece por 'En tramit...'
    {
        "$match": {
            "licencias.desc_tipo_situacion_licencia": {
                "$regex": re.compile(r"en\s+tr[aá]mit", re.IGNORECASE)
            }
        }
    },
    {"$unwind": "$licencias"},
    {
        "$match": {
            "licencias.desc_tipo_situacion_licencia": {
                "$regex": re.compile(r"en\s+tr[aá]mit", re.IGNORECASE)
            }
        }
    },
    {
        "$project": {
            "_id":             0,
            "id_local":        1,
            "rotulo":          1,
            "distrito":        "$desc_distrito_local",
            "barrio":          "$desc_barrio_local",
            "ref_licencia":    "$licencias.ref_licencia",
            "tipo_licencia":   "$licencias.desc_tipo_licencia",
            "estado_licencia": "$licencias.desc_tipo_situacion_licencia",
            "tiene_terraza":   {"$cond": [{"$ne": ["$terraza.id_terraza", None]}, True, False]}
        }
    },
    {"$limit": 10}
]

resultados_4c = list(col_locales.aggregate(pipeline_4c))
print(f"Locales/terrazas con licencia 'En trámite/tramitación' (muestra de {len(resultados_4c)}):")
for r in resultados_4c:
    print(f"  [{r['id_local']}] {r.get('rotulo','—'):<30} | {r['distrito']:<20} | {r['barrio']:<20} | {r['tipo_licencia']:<40} | {r['ref_licencia']} | terraza: {r['tiene_terraza']}")

Locales/terrazas con licencia 'En trámite/tramitación' (muestra de 10):
  [270543034] HERBOLARIO                     | USERA                | ALMENDRALES          | Declaración Responsable                  | 500/2013/03271 | terraza: False
  [270543034] HERBOLARIO                     | USERA                | ALMENDRALES          | Transmisión de licencia Urbanística      | 220/2013/11467 | terraza: False
  [270543034] HERBOLARIO                     | USERA                | ALMENDRALES          | Declaración Responsable                  | 500/2015/14921 | terraza: False
  [270544370] ALIMENTACION                   | VILLA DE VALLECAS    | CASCO H.VALLECAS     | Transmisión de licencia Urbanística      | 220/2013/11972 | terraza: False
  [280047510] NATURHOUSE                     | FUENCARRAL-EL PARDO  | EL PILAR             | Declaración Responsable                  | 500/2015/15219 | terraza: False
  [270527596] LOCUTORIO TRANSFER LATINA      | CARABANCHEL          | VISTA ALEGRE      

### 4d. Consulta por sección, división y epígrafe de la actividad comercial

In [44]:
pipeline_4d_general = [
    {"$unwind": {"path": "$actividades", "preserveNullAndEmptyArrays": False}},
    
    {
        "$match": {
            "$and": [
                {"actividades.id_seccion": {"$ne": None}},
                {"actividades.id_division": {"$ne": None}},
                {"actividades.id_epigrafe": {"$ne": None}},
            ]
        }
    },
    
    {
        "$group": {
            "_id": {
                "seccion":   "$actividades.id_seccion",
                "desc_sec":  "$actividades.desc_seccion",
                "division":  "$actividades.id_division",
                "desc_div":  "$actividades.desc_division",
                "epigrafe":  "$actividades.id_epigrafe",
                "desc_epi":  "$actividades.desc_epigrafe"
            },
            "total_locales":  {"$sum": 1},
            "total_terrazas": {
                "$sum": {"$cond": [{"$ne": ["$terraza.id_terraza", None]}, 1, 0]}
            }
        }
    },
    
    {"$sort": {"total_locales": -1}},
    {"$limit": 10}
]

resultados_4d_general = list(col_locales.aggregate(pipeline_4d_general))

print("Clasificación general por Sección, División y Epígrafe (top 10):")
for r in resultados_4d_general:
    g = r['_id']
    print(f"  [{g.get('seccion')}/{g.get('division')}/{g.get('epigrafe')}] {g.get('desc_epi', '—'):<105} | Locales: {r.get('total_locales'):>4} | Terrazas: {r.get('total_terrazas'):>3}")

Clasificación general por Sección, División y Epígrafe (top 10):
  [I/56/561004] BAR RESTAURANTE                                                                                           | Locales: 2379 | Terrazas: 1079
  [I/56/561005] BAR CON COCINA                                                                                            | Locales: 2106 | Terrazas: 921
  [S/96/960201] SERVICIO DE PELUQUERIA                                                                                    | Locales: 1707 | Terrazas:   1
  [I/56/561006] CAFETERIA                                                                                                 | Locales: 1407 | Terrazas: 623
  [G/47/477101] COMERCIO AL POR MENOR DE PRENDAS DE VESTIR EN ESTABLECIMIENTOS ESPECIALIZADOS                             | Locales: 1351 | Terrazas:   1
  [I/56/561001] RESTAURANTE                                                                                               | Locales: 1202 | Terrazas: 436
  [S/96/96

### 4e. Actividad económica más frecuente por barrio y distrito

In [ ]:
pipeline_4e = [
    {"$unwind": {"path": "$actividades", "preserveNullAndEmptyArrays": False}},
    
    {
        "$group": {
            "_id": {
                "distrito":  "$desc_distrito_local",
                "barrio":    "$desc_barrio_local",
                "actividad": "$actividades.desc_epigrafe"
            },
            "frecuencia": {"$sum": 1}
        }
    },

    {"$sort": {"frecuencia": -1}},
    
    {
        "$group": {
            "_id": {
                "distrito": "$_id.distrito",
                "barrio":   "$_id.barrio"
            },
            "actividad_predominante": {"$first": "$_id.actividad"},
            "frecuencia":             {"$first": "$frecuencia"}
        }
    },
    {"$sort": {"_id.distrito": 1, "_id.barrio": 1}},
    {"$limit": 10},
    {
        "$project": {
            "_id":                    0,
            "distrito":               "$_id.distrito",
            "barrio":                 "$_id.barrio",
            "actividad_predominante": 1,
            "frecuencia":             1
        }
    }
]

resultados_4e = list(col_locales.aggregate(pipeline_4e))
print("Actividad económica más frecuente por barrio/distrito (top 10):")
for r in resultados_4e:
    distrito = r.get('distrito') or 'Sin distrito'
    barrio = r.get('barrio') or 'Sin barrio'
    actividad = r.get('actividad_predominante') or 'Sin actividad'
    frecuencia = r.get('frecuencia', 0)
    
    print(f"  {distrito:<22} | {barrio:<28} | {actividad:<40} ({frecuencia})")
   

Actividad económica más frecuente por barrio/distrito (top 10):
  ARGANZUELA             | ACACIAS                      | Sin actividad                            (25)
  ARGANZUELA             | ATOCHA                       | COMERCIO AL POR MENOR DE TABACOS Y ARTICULOS DE FUMADOR (2)
  ARGANZUELA             | CHOPERA                      | Sin actividad                            (35)
  ARGANZUELA             | DELICIAS                     | BAR CON COCINA                           (32)
  ARGANZUELA             | IMPERIAL                     | Sin actividad                            (24)
  ARGANZUELA             | LEGAZPI                      | Sin actividad                            (22)
  ARGANZUELA             | PALOS DE LA FRONTERA         | SERVICIO DE PELUQUERIA                   (28)
  BARAJAS                | AEROPUERTO                   | Sin actividad                            (12)
  BARAJAS                | ALAMEDA DE OSUNA             | ACTIVIDADES ADMINISTRATIVAS Y AU

### 4f. Actualización de horarios de apertura y cierre

**Criterio elegido:** Actualizar el horario de los locales del barrio **"SAN ANDRES"** cuya actividad principal sea hostelería (sección I), estableciendo el horario estándar de tarde: apertura `13:00`, cierre `01:00`.

**Justificación:** Barrio con más locales.

In [59]:
# Primero, contar cuántos documentos cumplen el criterio
filtro_4f = {
    "desc_barrio_local":   {"$regex": re.compile(r"san andres", re.IGNORECASE)},
    "actividades.id_seccion": "I"
}

total_afectados = col_locales.count_documents(filtro_4f)
print(f"Documentos que cumplen el criterio: {total_afectados}")

Documentos que cumplen el criterio: 43


In [60]:
# Ejecutar la actualización
resultado_update = col_locales.update_many(
    filter=filtro_4f,
    update={
        "$set": {
            "hora_apertura1": "13:00",
            "hora_cierre2":   "01:00"
        }
    }
)

print(f"Documentos encontrados : {resultado_update.matched_count}")
print(f"Documentos modificados : {resultado_update.modified_count}")

# Verificar uno de los documentos actualizados
doc_verificacion = col_locales.find_one(
    filtro_4f,
    {"rotulo": 1, "desc_barrio_local": 1, "hora_apertura1": 1, "hora_cierre2": 1}
)
print("\nEjemplo de documento actualizado:")
print(json.dumps(doc_verificacion, ensure_ascii=False, indent=2, default=str))

Documentos encontrados : 43
Documentos modificados : 43

Ejemplo de documento actualizado:
{
  "_id": "6996ee2c057e2678f3de2cb9",
  "desc_barrio_local": "SAN ANDRES",
  "rotulo": "PANYVINO RESTAURANTE",
  "hora_apertura1": "13:00",
  "hora_cierre2": "01:00"
}


## 5. Creación y uso de índices

### 5a. Índice simple sobre `desc_barrio_local`

In [66]:
# Plan de ejecución SIN índice
query_barrio = {"desc_barrio_local": "SAN ANDRES"}

plan_sin_idx = col_locales.find(query_barrio).explain()
winning_plan_sin = plan_sin_idx.get('queryPlanner', {}).get('winningPlan', {})
stats_sin = plan_sin_idx.get('executionStats', {})
print("=== SIN ÍNDICE ===")
print(f"  Stage       : {winning_plan_sin.get('stage', '—')}")
print(f"  Docs examinados : {stats_sin.get('totalDocsExamined', '—')}")
print(f"  Tiempo (ms)     : {stats_sin.get('executionTimeMillis', '—')}")

=== SIN ÍNDICE ===
  Stage       : COLLSCAN
  Docs examinados : 27402
  Tiempo (ms)     : 12


In [69]:
# Crear índice simple
idx_barrio = col_locales.create_index([("desc_barrio_local")], name="idx_barrio")
print(f"Índice creado: {idx_barrio}")

# Plan de ejecución CON índice
plan_con_idx = col_locales.find(query_barrio).explain()
winning_plan_con = plan_con_idx.get('queryPlanner', {}).get('winningPlan', {})
stats_con = plan_con_idx.get('executionStats', {})
print("\n=== CON ÍNDICE SIMPLE (desc_barrio_local) ===")
print(f"  Stage       : {winning_plan_con.get('stage', '—')}")
print(f"  Docs examinados : {stats_con.get('totalDocsExamined', '—')}")
print(f"  Tiempo (ms)     : {stats_con.get('executionTimeMillis', '—')}")

Índice creado: idx_barrio

=== CON ÍNDICE SIMPLE (desc_barrio_local) ===
  Stage       : FETCH
  Docs examinados : 831
  Tiempo (ms)     : 1


### 5b. Índice compuesto sobre `desc_distrito_local` + `desc_barrio_local`

In [71]:
idx_dist_barrio = col_locales.create_index(
    [("desc_distrito_local", pymongo.ASCENDING),
     ("desc_barrio_local",   pymongo.ASCENDING)],
    name="idx_distrito_barrio"
)
print(f"Índice compuesto creado: {idx_dist_barrio}")

query_compuesta = {"desc_distrito_local": "CENTRO", "desc_barrio_local": "EMBAJADORES"}
plan_comp = col_locales.find(query_compuesta).explain()
winning = plan_comp.get('queryPlanner', {}).get('winningPlan', {})
stats   = plan_comp.get('executionStats', {})

print("\n=== CON ÍNDICE COMPUESTO (distrito + barrio) ===")
print(f"  Stage           : {winning.get('stage', '—')}")
print(f"  Docs examinados : {stats.get('totalDocsExamined', '—')}")
print(f"  Tiempo (ms)     : {stats.get('executionTimeMillis', '—')}")

Índice compuesto creado: idx_distrito_barrio

=== CON ÍNDICE COMPUESTO (distrito + barrio) ===
  Stage           : FETCH
  Docs examinados : 638
  Tiempo (ms)     : 1


### 5c. Índice de array sobre `actividades.desc_epigrafe`

In [75]:
idx_actividades = col_locales.create_index([("actividades.desc_epigrafe")], name="idx_actividades_epigrafe")
print(f"Índice de array creado: {idx_actividades}")

query_array = {"actividades.desc_epigrafe": "TAPICERIA"}
plan_array  = col_locales.find(query_array).explain()
winning_arr = plan_array.get('queryPlanner', {}).get('winningPlan', {})
stats_arr   = plan_array.get('executionStats', {})

print("\n=== CON ÍNDICE DE ARRAY (actividades.desc_epigrafe) ===")
print(f"  Stage           : {winning_arr.get('stage', '—')}")
print(f"  Docs examinados : {stats_arr.get('totalDocsExamined', '—')}")
print(f"  Tiempo (ms)     : {stats_arr.get('executionTimeMillis', '—')}")

Índice de array creado: idx_actividades_epigrafe

=== CON ÍNDICE DE ARRAY (actividades.desc_epigrafe) ===
  Stage           : FETCH
  Docs examinados : 47
  Tiempo (ms)     : 0


## 6. Modelo de datos — Versión 2: Extensión con alojamientos turísticos

### 6.1 Revisión del dataset Airbnb

El dataset `airbnb_listings.json` ya está disponible localmente en `data/raw/`.  
Fue procesado en `01_data_preparation.ipynb` y exportado como `listings_mongodb.json` en `data/processed/`.

**Decisión de modelado:** Los listings se almacenan en una **colección independiente** (`listings`) relacionada con los locales por el campo `neighbourhood_group_cleansed` ↔ `desc_distrito_local`.  
Se elige colección separada porque los alojamientos tienen identidad propia y consultas diferenciadas.

### 6.2 Consultas del modelo v2

#### Consulta v2-a: Total de alojamientos, locales y terrazas por distrito y barrio

In [ ]:
# Paso 1: totales de locales y terrazas por distrito (clave normalizada a minúsculas)
pipe_locales_dist = [
    {
        "$group": {
            "_id":              {"$toLower": "$desc_distrito_local"},
            "distrito_original": {"$first": "$desc_distrito_local"},
            "total_locales":    {"$sum": 1},
            "total_terrazas":   {
                "$sum": {"$cond": [{"$ne": ["$terraza.id_terraza", None]}, 1, 0]}
            }
        }
    }
]
dict_locales = {
    r["_id"]: r
    for r in col_locales.aggregate(pipe_locales_dist)
    if r["_id"]
}

# Paso 2: totales de listings por distrito (clave normalizada a minúsculas)
pipe_listings_dist = [
    {
        "$group": {
            "_id":               {"$toLower": "$neighbourhood_group_cleansed"},
            "distrito_original": {"$first": "$neighbourhood_group_cleansed"},
            "total_alojamientos": {"$sum": 1},
            "precio_medio":       {"$avg": "$price"}
        }
    }
]
dict_listings = {
    r["_id"]: r
    for r in col_listings.aggregate(pipe_listings_dist)
    if r["_id"]
}

# Paso 3: join - la clave normalizada coincide en ambos lados
distritos = set(dict_locales.keys()) | set(dict_listings.keys())
resumen = []
for key in sorted(distritos):
    loc = dict_locales.get(key, {})
    lst = dict_listings.get(key, {})
    nombre = lst.get("distrito_original") or loc.get("distrito_original") or key
    resumen.append({
        "distrito":            nombre,
        "total_locales":       loc.get("total_locales", 0),
        "total_terrazas":      loc.get("total_terrazas", 0),
        "total_alojamientos":  lst.get("total_alojamientos", 0),
        "precio_medio_airbnb": round(lst.get("precio_medio", 0) or 0, 2),
    })

import pandas as pd
df_v2a = pd.DataFrame(resumen).sort_values("total_alojamientos", ascending=False)
print("Total de alojamientos, locales y terrazas por distrito (join por distrito normalizado):")
display(df_v2a)

Total de alojamientos, locales y terrazas por distrito (join por distrito normalizado):


,distrito,total_locales,total_terrazas,total_alojamientos,precio_medio_airbnb
3,Centro,2349,145,8474,81.27
18,Salamanca,1499,127,1096,99.78
7,Chamberí,0,0,1026,85.98
0,Arganzuela,1006,68,944,62.64
22,Tetuán,0,0,584,58.23
17,Retiro,749,64,536,83.44
13,Moncloa - Aravaca,0,0,497,84.29
2,Carabanchel,2398,70,457,38.74
5,Chamartín,0,0,430,84.79
12,Latina,1830,69,421,43.31


#### Consulta v2-b: Barrios con mayor número de alojamientos y terrazas con licencias concedidas en los últimos 2 años

In [27]:
# Fecha límite: 2 años atrás desde la fecha de los datos (diciembre 2023)
fecha_limite = "2021-12-01"

# Terrazas con licencia concedida en los últimos 2 años, agrupadas por barrio.
# Se extrae también distrito_key (minúsculas) para cruzar con listings.
pipeline_v2b_terrazas = [
    {
        "$match": {
            "terraza.id_terraza": {"$ne": None}, 
            "licencias": {
                "$elemMatch": {
                    "desc_tipo_situacion_licencia": {
                        "$regex": "concedida",
                        "$options": "i"
                    },
                    "fecha_dec_lic": {"$gte": fecha_limite}
                }
            }
        }
    },
    {
        "$group": {
            "_id": {
                "distrito_key": {"$toLower": "$desc_distrito_local"},
                "distrito":     "$desc_distrito_local",
                "barrio":       "$desc_barrio_local"
            },
            "terrazas_licencia_reciente": {"$sum": 1}
        }
    },
    {"$sort": {"terrazas_licencia_reciente": -1}},
    {"$limit": 10}
]

resultados_terrazas = list(col_locales.aggregate(pipeline_v2b_terrazas))

# Alojamientos por distrito, clave normalizada a minúsculas para que el cruce funcione
pipeline_v2b_listings = [
    {
        "$group": {
            "_id":                {"$toLower": "$neighbourhood_group_cleansed"},
            "total_alojamientos": {"$sum": 1}
        }
    }
]
dict_aloj_por_distrito = {
    r["_id"]: r["total_alojamientos"]
    for r in col_listings.aggregate(pipeline_v2b_listings)
    if r["_id"]
}

print("Barrios con más terrazas con licencia concedida (últimos 2 años) + alojamientos del distrito:")
print(f"{'Distrito':<22} {'Barrio':<28} {'Terrazas recientes':>18} {'Alojamientos':>14}")
print("-" * 85)
for r in resultados_terrazas:
    g = r["_id"]
    aloj = dict_aloj_por_distrito.get(g["distrito_key"], 0)
    print(f"{g['distrito']:<22} {g['barrio']:<28} {r['terrazas_licencia_reciente']:>18} {aloj:>14}")

Barrios con más terrazas con licencia concedida (últimos 2 años) + alojamientos del distrito:
Distrito               Barrio                       Terrazas recientes   Alojamientos
-------------------------------------------------------------------------------------
CENTRO                 PALACIO                                      13           8474
LATINA                 ALUCHE                                       12            421
CENTRO                 SOL                                          11           8474
TETUAN                 CUATRO CAMINOS                               10              0
SALAMANCA              GOYA                                         10           1096
CENTRO                 EMBAJADORES                                   9           8474
ARGANZUELA             ACACIAS                                       9            944
CHAMBERI               TRAFALGAR                                     9              0
FUENCARRAL-EL PARDO    VALVERDE               